# Ingestion — Piece 1: Load & Inspect the WHO PDF

Loads the WHO hypertension guideline PDF page-by-page using `PyPDFLoader`
and inspects what metadata/content it gives us for free, before we add our
own citation metadata (title, publisher, source URL) in the next piece.

In [ ]:
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader


## Load the PDF

In [19]:
file_path = Path("..") / "data" / "raw" / "who_hypertension_2021.pdf"
loader = PyPDFLoader(file_path)

who_docs = loader.load()

print(f"Loaded {len(who_docs)} pages from {file_path}")

Loaded 61 pages from ../data/raw/who_hypertension_2021.pdf


## Inspect the first page's metadata

See what `PyPDFLoader` already gives us for free — page index, page label
(the PDF's own printed page numbering), source path, total page count —
before we stamp our own title/publisher/URL onto it in the next piece.

In [20]:
print(f"Page 0 metadata: {who_docs[0].metadata}")

Page 0 metadata: {'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2022-02-11T15:30:29+00:00', 'moddate': '2022-11-30T11:03:18+00:00', 'trapped': '/False', 'source': '../data/raw/who_hypertension_2021.pdf', 'total_pages': 61, 'page': 0, 'page_label': 'a'}


## Inspect page content

Preview raw extracted text on a page with real content (not the near-empty
cover page) to sanity-check the extraction quality.

In [21]:
# Fourth page (index 3) — the cover page (index 0) has too little text to be a useful preview
print(f"Page label: {who_docs[3].metadata['page_label']}")
print(who_docs[3].page_content[:1000])

Page label: ii
Guideline for the pharmacological treatment of hypertension in adults
ISBN 978-92-4-003398-6 (electronic version) 
ISBN 978-92-4-003397-9 (print version)
© World Health Organization 2021
Some rights reserved. This work is available under the Creative Commons Attribution-NonCommercial-
ShareAlike 3.0 IGO licence (CC BY-NC-SA 3.0 IGO;  
https://creativecommons.org/licenses/by-nc-sa/3.0/igo). 
Under the terms of this licence, you may copy, redistribute and adapt the work for non-commercial 
purposes, provided the work is appropriately cited, as indicated below. In any use of this work, there 
should be no suggestion that WHO endorses any specific organization, products or services. The use of 
the WHO logo is not permitted. If you adapt the work, then you must license your work under the same 
or equivalent Creative Commons licence. If you create a translation of this work, you should add the 
following disclaimer along with the suggested citation: “This translation was not

### Next: Piece 2 — stamp `title`, `publisher`, and `source URL` onto every document's metadata, since `PyPDFLoader` doesn't give us those for free.

In [22]:
# These three values are constant for every page of this PDF, so we define
# them once here rather than repeating them inside the loop below.
# PyPDFLoader doesn't know any of this — it only extracted what's embedded
# in the PDF file itself (producer, creator, source path, page, page_label).
title = "Guideline for the pharmacological treatment of hypertension in adults"
publisher = "World Health Organization"
source_url = "https://iris.who.int/server/api/core/bitstreams/f062769d-f075-4a00-87af-0a2106e0bd04/content"

# who_docs is a list of Document objects (one per PDF page), and each one's
# .metadata is just a plain dict — the same dict you printed in Piece 1.
# We loop over the list and mutate each dict in place with .update(),
# rather than building a new list, because we're not replacing the
# Documents, just adding keys to metadata they already have.
for doc in who_docs:
    doc.metadata.update({
        "title": title,
        "publisher": publisher,
        "source_url": source_url,
    })

# Sanity check: page 3's metadata should now show title/publisher/source_url
# alongside the page/page_label keys PyPDFLoader already gave us.
print(who_docs[3].metadata)

{'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2022-02-11T15:30:29+00:00', 'moddate': '2022-11-30T11:03:18+00:00', 'trapped': '/False', 'source': '../data/raw/who_hypertension_2021.pdf', 'total_pages': 61, 'page': 3, 'page_label': 'ii', 'title': 'Guideline for the pharmacological treatment of hypertension in adults', 'publisher': 'World Health Organization', 'source_url': 'https://iris.who.int/server/api/core/bitstreams/f062769d-f075-4a00-87af-0a2106e0bd04/content'}


# Ingestion — CDC PDF

Same load + metadata-stamp pattern as the WHO section above, applied to the
CDC Million Hearts *Hypertension Control Change Package*.

In [23]:
file_path = Path("..") / "data" / "raw" / "cdc_million_hearts_change_package.pdf"
loader = PyPDFLoader(file_path)

cdc_docs = loader.load()

print(f"Loaded {len(cdc_docs)} pages from {file_path}")

Loaded 30 pages from ../data/raw/cdc_million_hearts_change_package.pdf


In [24]:
print(f"Page 0 metadata: {cdc_docs[0].metadata}")

Page 0 metadata: {'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.0 (Macintosh)', 'creationdate': '2020-05-18T10:52:16-04:00', '508 compliance': 'Palladian Partners', 'author': 'Centers for Disease Control and Prevention', 'keywords': 'DHHS; CDC; Hypertension Control; Change Package', 'moddate': '2025-05-29T14:42:44-04:00', 'subject': 'Hypertension Control Change Package, 2nd edition', 'title': 'Hypertension Control Change Package', 'trapped': '/Unknown', 'source': '../data/raw/cdc_million_hearts_change_package.pdf', 'total_pages': 30, 'page': 0, 'page_label': 'i'}


In [25]:
# Fourth page (index 3) — the cover page (index 0) has too little text to be a useful preview
print(f"Page label: {cdc_docs[3].metadata['page_label']}")
print(cdc_docs[3].page_content[:1000])

Page label: 1
CHANGE PACKAGE  | 1 
Hypertension Control  Change Package — Quick Reference 
Focus Areas 
Key 
Foundations 
Equipping Care 
Teams 
Population Health 
Management 
Individual Patient 
Supports 
 
 
 
 
  
Change Concepts and Change Ideas 
Key Foundations 
Make HTN Control a Practice Priority 
Designate a practice or health system champion, such as a head physician or quality improvement lead  
Ensure care team engagement in HTN control 
Redesign office or exam space to support proper BP measurement technique 
Provide BP checks without appointment or co-pay 
Expand the HTN care team with community pharmacists and/or community health workers 
Implement a Policy or Process to Address BP for Every Patient with HTN at Every Visit 
Develop HTN control policies and procedures 
Develop a flowchart/workflow for proactively tracking and managing patients with HTN 
Deploy HTN treatment protocols and algorithms 
Overcome diagnostic and treatment inertia 
Manage resistant HTN 
Evaluate 

In [26]:
# Same pattern as the WHO cell above — see its comments for the why.
title = "Hypertension Control Change Package"
publisher = "Centers for Disease Control and Prevention (Million Hearts)"
source_url = "https://stacks.cdc.gov/view/cdc/251804"

for doc in cdc_docs:
    doc.metadata.update({
        "title": title,
        "publisher": publisher,
        "source_url": source_url,
    })

print(cdc_docs[3].metadata)

{'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.0 (Macintosh)', 'creationdate': '2020-05-18T10:52:16-04:00', '508 compliance': 'Palladian Partners', 'author': 'Centers for Disease Control and Prevention', 'keywords': 'DHHS; CDC; Hypertension Control; Change Package', 'moddate': '2025-05-29T14:42:44-04:00', 'subject': 'Hypertension Control Change Package, 2nd edition', 'title': 'Hypertension Control Change Package', 'trapped': '/Unknown', 'source': '../data/raw/cdc_million_hearts_change_package.pdf', 'total_pages': 30, 'page': 3, 'page_label': '1', 'publisher': 'Centers for Disease Control and Prevention (Million Hearts)', 'source_url': 'https://stacks.cdc.gov/view/cdc/251804'}


# Ingestion — USPSTF PDF

Same pattern once more, for the USPSTF hypertension screening
reaffirmation recommendation statement.

In [27]:
file_path = Path("..") / "data" / "raw" / "uspstf_hypertension_2021.pdf"
loader = PyPDFLoader(file_path)

uspstf_docs = loader.load()

print(f"Loaded {len(uspstf_docs)} pages from {file_path}")

Loaded 7 pages from ../data/raw/uspstf_hypertension_2021.pdf


In [28]:
print(f"Page 0 metadata: {uspstf_docs[0].metadata}")

Page 0 metadata: {'producer': 'PDF generator', 'creator': 'XyEnterprise XPP 9.2.2.0', 'creationdate': '2021-04-16T08:37:37-05:00', 'author': 'U.S. Preventive Services Task Force', 'keywords': 'high blood pressure, HBP, hypertension, adults', 'moddate': '2021-04-23T08:58:05-04:00', 'title': 'Screening for Hypertension in Adults: US Preventive Services Task Force Reaffirmation Recommendation Statement', 'source': '../data/raw/uspstf_hypertension_2021.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}


In [29]:
# Fourth page (index 3) — a preview of real body content, a few pages into the recommendation statement
print(f"Page label: {uspstf_docs[3].metadata['page_label']}")
print(uspstf_docs[3].page_content[:1000])

Page label: 4
• Screening for atrial fibrillation with electrocardiography13
• Behavioral counseling interventions to promote a healthy diet and
physical activity for cardiovascular disease prevention
• In adults with cardiovascular risk factors14
• In adults without known cardiovascular risk factors15
• Statin use for the primary prevention of cardiovascular disease
in adults16
• Aspirin use to prevent cardiovascular disease and colorectal cancer17
• Screening for high blood pressure in children and adolescents18
Reaffirmation of Previous USPSTF
Recommendation
This recommendation is a reaffirmation of the 2015 recommenda-
tion statement on screening for high blood pressure in adults (A rec-
ommendation). The USPSTF has issued an A recommendation on
screening for high blood pressure in adults since 1996 (updated in
2003, reaffirmed in 2007 , and then updated in 2015). In 2015, the
USPSTF recommended screening for high blood pressure in adults
18 years or older and obtaining measurement

In [30]:
# Same pattern as the WHO cell above — see its comments for the why.
title = "Screening for Hypertension in Adults"
publisher = "U.S. Preventive Services Task Force"
source_url = "https://www.uspreventiveservicestaskforce.org/uspstf/recommendation/hypertension-in-adults-screening"

for doc in uspstf_docs:
    doc.metadata.update({
        "title": title,
        "publisher": publisher,
        "source_url": source_url,
    })

print(uspstf_docs[3].metadata)

{'producer': 'PDF generator', 'creator': 'XyEnterprise XPP 9.2.2.0', 'creationdate': '2021-04-16T08:37:37-05:00', 'author': 'U.S. Preventive Services Task Force', 'keywords': 'high blood pressure, HBP, hypertension, adults', 'moddate': '2021-04-23T08:58:05-04:00', 'title': 'Screening for Hypertension in Adults', 'source': '../data/raw/uspstf_hypertension_2021.pdf', 'total_pages': 7, 'page': 3, 'page_label': '4', 'publisher': 'U.S. Preventive Services Task Force', 'source_url': 'https://www.uspreventiveservicestaskforce.org/uspstf/recommendation/hypertension-in-adults-screening'}


## Next: Piece 3 — combine `who_docs`, `cdc_docs`, and `uspstf_docs` into one corpus and chunk it for embedding.

In [31]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Concatenate the three per-doc page lists into one corpus. Order doesn't
# matter for retrieval — FAISS will index every chunk regardless of which
# source document it came from.
all_docs = who_docs + cdc_docs + uspstf_docs

# chunk_size chosen by inspecting the WHO PDF directly: its recommendation
# pages (e.g. page index 18, "3.1 Blood pressure threshold...") run
# ~1600-4000 characters each. chunk_size=4000 keeps a single page's content
# in one chunk in most cases, so a recommendation that fits on one page
# won't get split mid-sentence by the splitter itself.
#
# Known limitation: PyPDFLoader gave us one Document per PDF page, and
# split_documents() chunks each Document independently — it never merges
# text across two Documents. So a recommendation that spans multiple PDF
# pages (3.1 continues from page 18 into page 19's "Evidence and rationale")
# still ends up in separate chunks at the page boundary, regardless of
# chunk_size. A large chunk_size avoids *extra* splits within a page; it
# can't undo a split that already exists between pages.
splitter = RecursiveCharacterTextSplitter(chunk_size=4000, chunk_overlap=400)

chunks = splitter.split_documents(all_docs)

print(f"{len(all_docs)} pages -> {len(chunks)} chunks")

98 pages -> 108 chunks


In [32]:
# Sanity check: one example chunk per source document, confirming the
# title/publisher/source_url/page_label metadata survived the split.
seen_titles = set()
for chunk in chunks:
    t = chunk.metadata["title"]
    if t not in seen_titles:
        seen_titles.add(t)
        print(f"--- {t} (page_label {chunk.metadata['page_label']}) ---")
        print(chunk.page_content[:300])
        print()

--- Guideline for the pharmacological treatment of hypertension in adults (page_label a) ---
Guideline 
for the
pharmacological 
treatment of 
hypertension 
in adults

--- Hypertension Control Change Package (page_label i) ---
Plan Act 
DoStudy 
A MILLION HEARTS ® ACTION GUIDE 
Hypertension 
Control 
CHANGE PACKAGE 
Second Edition

--- Screening for Hypertension in Adults (page_label 1) ---
Screening for Hypertension in Adults
US Preventive Services Task Force Reaffirmation
Recommendation Statement
US Preventive Services Task Force
Summary of Recommendation
See the Summary of Recommendation figure.
Importance
Hypertension is a prevalent condition, affects approximately 45%
of the adult



### Piece 4 — Set up embeddings (`nomic-embed-text` via Ollama) Wires up the embedding model and sanity-checks that it can reach the local Ollama server before Piece 5 uses it to build the FAISS index.

In [33]:
# Same langchain_community family as PyPDFLoader in Piece 1 — expect the
# same deprecation warning, harmless for our purposes tonight.
from langchain_community.embeddings import OllamaEmbeddings

# model= must match a model you've already pulled locally (`ollama list`
# showed nomic-embed-text) — this doesn't download anything, it just tells
# LangChain which local Ollama model to call.
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# .embed_query() sends one string to the local Ollama server and gets back
# its embedding vector (a list of floats). We're not embedding all 108
# chunks here — that's Piece 5's job, via FAISS.from_documents(), which
# calls .embed_documents() internally. This is just a connectivity check.
sample_vector = embeddings.embed_query(chunks[0].page_content)

# nomic-embed-text produces 768-dimensional vectors — printing the length
# confirms both that the call succeeded and what dimensionality to expect
# once FAISS builds the full index in Piece 5.
print(f"Embedding dimension: {len(sample_vector)}")

Embedding dimension: 768


# Piece 5 — Build & save the FAISS index

Embeds all chunks, builds the FAISS index, saves it to disk, then reloads
it and runs one similarity search as an end-to-end sanity check — this
closes out Day 1.

In [34]:
from langchain_community.vectorstores import FAISS

# FAISS.from_documents() calls embeddings.embed_documents() on every
# chunk's page_content under the hood, then builds a similarity-searchable
# index over the resulting vectors — one call replaces what would
# otherwise be a manual embed-loop + index-construction step.
vectorstore = FAISS.from_documents(chunks, embeddings)

# save_local() writes the FAISS index plus a pickle of the associated
# Documents (so page_content + metadata come back together on reload) to
# a folder — this means Day 2's retrieval step can load the index
# directly instead of re-embedding all 108 chunks every run.
index_path = Path("..") / "data" / "faiss_index"
vectorstore.save_local(str(index_path))

print(f"Saved FAISS index with {vectorstore.index.ntotal} vectors to {index_path}")

Saved FAISS index with 108 vectors to ../data/faiss_index


## Round-trip sanity check

Reload the saved index from disk (rather than reusing the in-memory
`vectorstore`) and run a real similarity search — this is a preview of what
Day 2's retrieval step will actually do.

In [35]:
# allow_dangerous_deserialization=True is required because loading a FAISS
# index uses pickle under the hood, which LangChain treats as unsafe by
# default (pickle can execute arbitrary code if the file came from
# somewhere untrusted). Safe here — it's the index we just wrote ourselves.
reloaded = FAISS.load_local(
    str(index_path), embeddings, allow_dangerous_deserialization=True
)

# similarity_search() embeds the query text with the same embeddings model,
# then returns the top-k chunks whose vectors are closest to it — exactly
# what Day 2's retrieval step will do for real user questions.
results = reloaded.similarity_search(
    "What blood pressure threshold should trigger starting medication?", k=2
)
for r in results:
    print(f"[{r.metadata['title']}, page {r.metadata['page_label']}]")
    print(r.page_content[:300])
    print()

[Guideline for the pharmacological treatment of hypertension in adults, page 7]
3 Recommendations
3.1 Blood pressure threshold for initiation of pharmacological treatment
1. RECOMMENDA TION ON BLOOD PRESSURE THRESHOLD FOR INITIATION OF 
PHARMACOLOGICAL TREATMENT
WHO recommends initiation of pharmacological antihypertensive treatment of individuals 
with a confirmed diagnosis of

[Guideline for the pharmacological treatment of hypertension in adults, page 10]
GUIDELINE FOR THE PHARMACOLOGICAL TREATMENT OF HYPERTENSION IN ADULTS
Evidence-to-decision considerations
There is uncertainty about patients’ values and preferences regarding the issue of testing before starting 
treatment for HTN. The cost of tests such as electrolytes, creatinine, lipid panel, gl



In [37]:
# Same call as before, but _with_score returns (Document, score) tuples
# instead of just Documents — score is a FAISS distance, so *lower* means
# more similar (opposite of a 0-1 similarity percentage).
scored_results = reloaded.similarity_search_with_score(
    "What blood pressure threshold should trigger starting medication?", k=2
)
for doc, score in scored_results:
    print(f"score={score:.4f}  [{doc.metadata['title']}, page {doc.metadata['page_label']}]")
    print(doc.page_content[:1000])
    print()

score=231.3538  [Guideline for the pharmacological treatment of hypertension in adults, page 7]
3 Recommendations
3.1 Blood pressure threshold for initiation of pharmacological treatment
1. RECOMMENDA TION ON BLOOD PRESSURE THRESHOLD FOR INITIATION OF 
PHARMACOLOGICAL TREATMENT
WHO recommends initiation of pharmacological antihypertensive treatment of individuals 
with a confirmed diagnosis of hypertension and systolic blood pressure of ≥140 mmHg or 
diastolic blood pressure of ≥90 mmHg.
Strong recommendation, moderate- to high-certainty evidence
WHO recommends pharmacological antihypertensive treatment of individuals with existing 
cardiovascular disease and systolic blood pressure of 130–139 mmHg.
Strong recommendation, moderate- to high-certainty evidence
WHO suggests pharmacological antihypertensive treatment of individuals without 
cardiovascular disease but with high cardiovascular risk, diabetes mellitus, or chronic kidney 
disease, and systolic blood pressure of 130–139 mmHg.
C